> **Read this first (updated 18 Nov 2026).** This notebook is the **first cleaning round (English only)**, which produced 2,000 cleaned posts on 6 Nov 2026. The team later added **Malay and Manglish posts** ("Manglish mode", `cleaning.languages: [en, ms, mixed]` in `config.yaml`), because many Malaysian users post in Malay or mix Malay and English, and an English-only analysis would miss (and under-state) their opinions. Only other languages and scripts are still removed.
>
> The **final dataset** used for all results is `data/grab_sentiment_clean.csv` (**2,015 posts**: 1,348 English, 530 Malay, 137 mixed). The model comparison is in `06_model_comparison.ipynb` and the charts are in `05_visualisation_and_findings.ipynb`.
>
> This notebook still runs top to bottom with the English-only setting, so you can see the original rules at work. Because a few rules were refined later, a re-run today keeps 2,002 posts instead of 2,000. Saving is switched off (`SAVE_OUTPUTS = False`), so the original files are not overwritten.

# 02 · Data Cleaning
**Input:** `data/interim/posts_raw_combined.csv` (all Reddit, X and Threads items from notebook 01)
**Output (the final dataset):** `data/processed/FINAL_grab_my_posts_clean_english.csv`

Raw social-media data is messy. The same tweet shows up under several searches, some posts are in
other languages, some use "grab" as an ordinary verb ("grab your tickets!"), and some replies say
nothing more than "thanks". This notebook removes those and tidies the text so it's ready for sentiment analysis.

| Step | What we do | Why |
|---|---|---|
| 1 | Remove empty, deleted and removed posts | Nothing to analyse |
| 2 | Remove duplicates | The same post found by two search terms would be counted twice |
| 3 | Remove posts **by** official Grab accounts and bots (e.g. AutoModerator) | They are brand or system messages, not public opinion |
| 4 | **Keep English only** | Team decision: the analysis focuses on English (Malay, Indonesian, Tagalog etc. are removed) |
| 5 | Keep only posts **about Grab in Malaysia** | Removes "grab" as a verb, other countries, Grab's share price, fan-club promotions |
| 6 | **Anonymise** and **normalise** the text | Privacy, and consistent text for the models |
| 7 | Remove **low-information** content, but keep meaningful short opinions | "thanks!" or "At subang jaya" has no sentiment, but "Grab so expensive" does |
| 8 | Remove repeated text | Copy-paste spam and cross-posts |

Each removed item gets a plain-English **reason**, so we can show exactly what was removed and why.
All the rules are in `src/clean.py`; the adjustable settings are under `cleaning:` in `config.yaml`.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import re
import pandas as pd
from src import collect, clean

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)

cfg = collect.load_config()
ccfg = cfg["cleaning"]
interim = collect.project_path(cfg["paths"]["interim"])
processed = collect.project_path(cfg["paths"]["processed"])
tables = collect.project_path(cfg["paths"]["tables"])

raw = pd.read_csv(interim / "posts_raw_combined.csv")
print(f"Raw items: {len(raw):,}")
pd.crosstab(raw["platform"], raw["is_comment"].map({False: "posts", True: "comments/replies"}),
            margins=True, margins_name="TOTAL")

Raw items: 4,782


is_comment,comments/replies,posts,TOTAL
platform,,,
reddit,1372,124,1496
threads,90,287,377
twitter,919,1990,2909
TOTAL,2381,2401,4782


In [2]:
# This notebook documents the FIRST cleaning round: English only (team decision 6 Nov 2026).
# The final dataset later used Manglish mode (en + ms + mixed); see the note at the top.
cfg["cleaning"]["languages"] = ["en"]
SAVE_OUTPUTS = False   # True would overwrite the saved round-1 dataset, audit and funnel table

## How the trickier rules work

**English only (step 4).** Language-detection software has no Malay model: it labels Malay as Indonesian,
and it often gets short or mixed "Manglish" sentences wrong. So we combine two checks:
1. the `langdetect` library's guess, and
2. a count of common English words (*the, is, and, to…*) versus common Malay/Indonesian words (*tak, nak, dah, lah, yang…*)
   and Tagalog words (*ang, mga, naman, po…*).

A post is kept as English if it reads as English **and** non-English words are at most a quarter of the English ones.
That keeps an English sentence ending in "lah", but removes a mostly-Malay one.

**About Grab (step 5).** A post must refer to Grab *the company*: GrabFood, Grab driver, "took a Grab", @GrabMY,
"Grab is so expensive" and so on. A plain "grab" used as a verb ("grab a coffee", "grab your tickets") doesn't count.
The **7 Threads posts the team added by hand** (`manually_added`) were checked by a person, so they skip
these two automatic relevance rules (one doesn't name a Malaysian place, for example). They still go through every other step.
**Reddit comments** count as on-topic when the Reddit thread they reply to is about Grab, even if the comment
doesn't repeat the word "Grab" (e.g. *"the app charged me twice"*).
X and Threads posts that mention another country (Philippines, Singapore, Indonesia…) or Grab's share price are removed,
unless they also mention a Malaysian place or RM prices.

**Low-information content (step 7).** Very short posts (under 3 words) are removed **unless** they contain an
opinion word (from the VADER sentiment word list, e.g. *worst, love, slow, expensive*) or an emoji. That way
"Grab sucks", "so expensive 😤" and "love GrabFood" are kept, while "Try indrive", "At subang jaya" and
"thanks bro" are removed.

In [3]:
kept, dropped, funnel = clean.clean_posts(raw, cfg)
print(f"Kept {len(kept):,} of {len(raw):,} items ({len(kept)/len(raw):.0%})")
funnel

Kept 2,002 of 4,782 items (42%)


,step,items_remaining,removed_at_this_step
0,Raw items collected,4782,0
1,empty / deleted / removed,4738,44
2,"duplicate (same post, several queries)",4704,34
3,official Grab account,4643,61
4,bot / moderator message,4625,18
5,not English (Malay / Indonesian),3953,672
6,not English (other language),3407,546
7,Reddit comment in a thread not about Grab,3278,129
8,does not mention Grab (or uses 'grab' as a verb),2182,1096
9,about another country or Grab's stock,2112,70


### Final dataset size
The project needs **at least 2,000** clean items (`cleaning.min_clean_items` in `config.yaml`).

In [4]:
display(clean.platform_breakdown(kept))
target = ccfg["min_clean_items"]
status = "OK" if len(kept) >= target else "BELOW TARGET: collect more (see notebook 01)"
print(f"Clean items: {len(kept):,} / target {target:,}: {status}")
print("Date range:", kept["created_at"].min().date(), "to", kept["created_at"].max().date())

is_comment,comments/replies,posts,TOTAL
platform,,,
reddit,1064,105,1169
threads,18,21,39
twitter,293,501,794
TOTAL,1375,627,2002


Clean items: 2,002 / target 2,000: OK
Date range: 2025-11-05 to 2026-11-05


## What was removed and why
Removed items per reason and platform, then 5 examples of each reason, so we can check the rules
aren't throwing away good posts.

In [5]:
display(pd.crosstab(dropped["drop_reason"], dropped["platform"], margins=True, margins_name="TOTAL"))
for reason, grp in dropped.groupby("drop_reason"):
    print(f"\n--- {reason} ({len(grp)}) ---")
    for _, r in grp.sample(min(5, len(grp)), random_state=1).iterrows():
        print(f"  [{r.platform}] " + re.sub(r"\s+", " ", r.text)[:150])

platform,reddit,threads,twitter,TOTAL
drop_reason,,,,
Reddit comment in a thread not about Grab,129,0,0,129
Threads post with no Malaysian context,0,9,0,9
about another country or Grab's stock,0,3,67,70
bot / moderator message,18,0,0,18
business advert / fan-club promotion,0,0,41,41
does not mention Grab (or uses 'grab' as a verb),15,215,866,1096
"duplicate (same post, several queries)",0,0,34,34
empty / deleted / removed,44,0,0,44
filler reply with no opinion,16,0,0,16



--- Reddit comment in a thread not about Grab (129) ---
  [reddit] Welcome to Modern Fkeries.
  [reddit] Some businesses really do be that way. It might be a business that grew physically and financially but not technologically. I guess they might need to
  [reddit] There is a train there, it would be more cost effective compared to what URL is offering. 30 AUD is I believe a bargain compared to what grab would of
  [reddit] KLIA Express & Transit begin operation at 5AM, cost about RM50 from KLIA to KL Sentral, usually arrive in 30 minutes. From KL Sentral, find MRT Muzium
  [reddit] Actually hamsters are pretty dumb. My kids owned 4. All 4 of them died very easily avoidable deaths.

--- Threads post with no Malaysian context (9) ---
  [threads] @user.ai any grab delivery voucher today
  [threads] The lrt stations shouldve closed at 1am instead of 12.30 am . now im stranded at the station w hella expensive grab prices 😭
  [threads] Fact is the rider cheated and robbed your food & mone

## Anonymisation and text normalisation
**Anonymisation:** personal usernames in the text (`@someone`, `u/someone`) become `@user` / `u/user`;
links become `URL`; email addresses and phone numbers become `EMAIL` / `PHONE`. Grab's own handles
(e.g. `@GrabMY`) are kept, because they are a company rather than a person and show who the post is addressed to.
The author is only ever stored as `author_hash` (a one-way code), never as a name.

The final file has **two cleaned versions of the text**, because the two sentiment methods in notebook 04 need different things:

| Column | Used for | What it looks like |
|---|---|---|
| `text_clean` | **VADER** (rule-based) and for humans reading/labelling posts | Anonymised; emojis written as words, e.g. `[pouting face]`; "soooo" → "soo"; "!!!!!!" → "!!!"; tidy quotes and spaces. **Capitals and ! are kept**, because VADER reads "TERRIBLE!!!" as stronger than "terrible". |
| `text_norm` | The **machine-learning** model (scikit-learn) | Also lower-case, punctuation removed, emojis as single tokens (e.g. `emoji_pouting_face`), and words reduced to their base form ("drivers" → "driver") |

Examples (posts with emojis, repeated letters or links):

In [6]:
has_extra = kept["text"].str.contains(r"URL|@user|[!?]{3,}|(?i:a{3,}|o{3,}|e{3,}|h{3,})", regex=True) | kept["text_clean"].str.contains(r"\[", regex=True)
examples = kept[has_extra].sample(8, random_state=7)
for _, r in examples.iterrows():
    print(f"[{r.platform}]")
    print("  original :", re.sub(r"\s+", " ", r.text)[:200])
    print("  text_clean:", r.text_clean[:200])
    print("  text_norm :", r.text_norm[:200], "\n")

[twitter]
  original : @user @user @GrabMY agree
  text_clean: @user @user @GrabMY agree
  text_norm : user user grabmy agree 

[twitter]
  original : It's giving their PBB days Grab food delivery scene. HHAHAHAHAHA AZRALPH DOUBLEMINT ELYU DATE AZRALPH DOUBLEMINT ELYU DATE AZRALPH DOUBLEMINT ELYU DATE URL
  text_clean: It's giving their PBB days Grab food delivery scene. HHAHAHAHAHA AZRALPH DOUBLEMINT ELYU DATE AZRALPH DOUBLEMINT ELYU DATE AZRALPH DOUBLEMINT ELYU DATE URL
  text_norm : its giving their pbb day grab food delivery scene hhahahahaha azralph doublemint elyu date azralph doublemint elyu date azralph doublemint elyu date url 

[twitter]
  original : Grab food delivery URL
  text_clean: Grab food delivery URL
  text_norm : grab food delivery url 

[twitter]
  original : Really happy about the new Women-Only Rides update from @Grab. Women in Klang Valley might find this especially useful for evening rides with Advance Booking. Hope this rolls out across Malaysia and b
  text_c

### Short opinions that were kept
These are under 3 words, but they still express an opinion, so they stay in the dataset:

In [7]:
short = kept[kept["n_words"] < ccfg["min_words"]]
print(f"{len(short)} short opinion posts kept")
short[["platform", "text_clean"]].head(15)

9 short opinion posts kept


,platform,text_clean
119,reddit,Fuck em!
612,reddit,"Yeah, food."
614,reddit,[sleepy face] [loudly crying face] So... hungry... [face with tears of joy] [rolling on the floor laughing]
736,reddit,Good comment
835,reddit,funny
1127,reddit,Scary af
1132,reddit,Quite sad
1517,twitter,@user @user @GrabMY reckless!
1964,twitter,@user @user @GrabMY agree


## Privacy check
Before saving, we confirm that no personal @usernames, links or raw usernames are left in the dataset.

In [8]:
all_text = pd.concat([kept["text"], kept["text_clean"]])
mentions = all_text.str.findall(r"(?<![\w@])@(\w+)").explode().dropna()
personal = mentions[~mentions.str.lower().str.match(r"^(user|grab\w*)$")]
checks = {
    "links remaining": int(all_text.str.contains(r"https?://|www\.", regex=True).sum()),
    "personal @usernames remaining": len(personal),
    "u/usernames remaining": int(all_text.str.contains(r"(?<![\w/])u/(?!user\b)\w{3,}", regex=True).sum()),
    "username column present": int(any(c in kept.columns for c in ["username", "author", "author_name"])),
}
display(pd.Series(checks, name="count"))
assert sum(checks.values()) == 0, "Privacy check failed: see counts above"
print("Privacy check passed.")

links remaining                  0
personal @usernames remaining    0
u/usernames remaining            0
username column present          0
Name: count, dtype: int64

Privacy check passed.


## Save the final dataset

In [9]:
final_path = processed / ccfg["output_file"]
if SAVE_OUTPUTS:   # off: keep the saved round-1 files
    kept.to_csv(final_path, index=False)
    dropped.to_csv(processed / "cleaning_removed_items_audit.csv", index=False)
    funnel.to_csv(tables / "cleaning_funnel.csv", index=False)
    print("Saved:", final_path.relative_to(PROJECT_ROOT), f"({len(kept):,} rows)")
else:
    print(f"Not saved (SAVE_OUTPUTS = False). This run kept {len(kept):,} posts; the saved round-1 file "
          f"{final_path.relative_to(PROJECT_ROOT)} has {len(pd.read_csv(final_path)):,}.")

Not saved (SAVE_OUTPUTS = False). This run kept 2,002 posts; the saved round-1 file data/processed/FINAL_grab_my_posts_clean_english.csv has 2,000.


### Columns in the final dataset
| Column | Meaning |
|---|---|
| `post_id`, `platform` | The item's ID on its platform (reddit / twitter / threads) |
| `is_comment`, `parent_post_id`, `subreddit` | Whether it's a comment/reply; for Reddit, the thread it belongs to and the subreddit |
| `query` | Which search term(s) found it |
| `created_at` | When it was posted (UTC) |
| `author_hash` | One-way code for the author (same author = same code; the name can't be recovered) |
| `url` | Link to the item (username-free form) |
| `likes`, `replies` | Engagement counts at collection time |
| `text` | Original text, anonymised (usernames, links, emails, phones masked) |
| `text_clean` | Cleaned text for VADER and human labelling |
| `text_norm` | Fully normalised text for the machine-learning model |
| `n_words` | Number of words |
| `lang_detected`, `lang_prob` | langdetect's language guess and its confidence |
| `actor_sentiment_score/label` | Reddit only: the scraper's own AFINN sentiment score (for comparison only) |

---
**Next step:** `03_manual_labelling.ipynb` gives **every** post in the cleaned dataset (no sampling) a label (positive / negative / neutral / irrelevant) following `data/LABELLING_GUIDE.md`, then removes the irrelevant posts.